In [1]:
# Packages Imported
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"
import pdfplumber
import chromadb
from sentence_transformers import SentenceTransformer
import pandas as pd
import streamlit as st
from chromadb.utils import embedding_functions
import openai

In [2]:
# Extracting the text from the pdf 
with pdfplumber.open("Something_Something.pdf") as pdf:
    full_text = ""
    for page in pdf.pages:
        page_text = page.extract_text()
        if page_text:
            full_text += page_text + "\n"

In [3]:
# Count total words in the full text
total_words = len(full_text.split())
print("Total words in movie text:", total_words)

Total words in movie text: 576


In [4]:
# Split text into chunks by the keyword "Scene"
chunks = full_text.split("Scene")

# Re-add the word "Scene" to each chunk and clean up
chunks = ["Scene" + chunk.strip() for chunk in chunks if chunk.strip()]


In [5]:
# Load the Embedding Model
embedding_function = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)

In [6]:
# Start ChromaDB and Create a Collection
chroma_client = chromadb.Client()
collection = chroma_client.get_or_create_collection(
    name="movie_script", 
    embedding_function=embedding_function
)

In [7]:
# Store Chunks into ChromaDB
for i, chunk in enumerate(chunks):
    collection.add(
        documents=[chunk],
        ids=[f"scene_{i}"]
    )

In [8]:
# creating the prompt 
def create_prompt(context, question):
    return f"Context: {context}\n\nQuestion: {question}\nAnswer:"


In [35]:
# Getting response from gpt
openai.api_key = "xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx"

client = openai.OpenAI(api_key=openai.api_key )  # new client-style

def get_gpt_answer(prompt):
    response = client.chat.completions.create(
        model="gpt-6.2-sol",
        messages=[
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": prompt}
        ],
        max_completion_tokens=200
    )
    return response.choices[0].message.content

In [ ]:
print("Ask anything about the movie! (type 'exit' to stop)\n")

while True:
    question = input("You: ")

    if question.lower() == "exit":
        print("Chat ended.")
        break

    results = collection.query(query_texts=[question], n_results=1)
    chunk = results["documents"][0][0]  # directly get the top scene

    print("\nRetrieved Chunk:\n", chunk)


    prompt = create_prompt(chunk, question)
    answer = get_gpt_answer(prompt)

    print("\nAnswer:", answer)
    print("-" * 50)


Ask anything about the movie! (type 'exit' to stop)



You:  who is kavitha ?



Retrieved Chunk:
 Scene2: Introduction to Muthupandi and Kavitha
The flashback begins in a small Tamil Nadu village. Muthupandi is a caring, hard-working brother
who raised his younger sister Kavitha (Trisha) after their mother died. They live modestly but with
strong family values and immense affection for each other.

Answer: Kavitha is Muthupandi’s younger sister, played by Trisha. Muthupandi raised her after their mother died.
--------------------------------------------------


You:  what relationship santhosh and kavitha share in the film



Retrieved Chunk:
 Scene4: Enter Santhosh The Hero
At the wedding, Kavitha meets Santhosh (Jayam Ravi), a rich, westernized NRI from London who is
also Lalithas cousin. He is fun-loving, talkative, and charming. He immediately becomes interested
in Kavitha.

Answer: Santhosh and Kavitha are love interests. They meet at the wedding, where Santhosh is immediately attracted to her.
--------------------------------------------------


You:  did sanjay the hero of this film 



Retrieved Chunk:
 Scene1: Police Station The Story Begins
The movie opens in a police station where Muthupandi (Prabhu), a rugged man from a village, is
arrested for murder. The police ask for his side of the story. He begins to narrate his past and the
events that led to the arrest.

Answer: The scene identifies Muthupandi, played by Prabhu, as the central character. It doesn’t mention Sanjay.
--------------------------------------------------
